# NB00 - Pipeline Utilities
**Eligibility & Accumulator**

In production, a reconciliation pipeline has dozens of checks spread across multiple notebooks.
If one those check fails, say the TPA eligibility table got dropped by an upstream job, you don't
want the entire pipeline to crash and leave you with nothing. You still want the accumulator
reconciliation, the copay audit, and the rollover checks to run and produce findings.

You also need to know exactly what ran, what passed, and what eventually broke.

This notebook provides shared utilities that every downstream notebook imports via
`%run ./NB_00_pipeline_utilities`. It handles three things:

| Utility | What it does |
|---------|-------------|
| Structured logging | JSON audit entries to `pbm_sim.pipeline_audit_log`, queryable by run, notebook, section |
| Error handling decorator | Wraps each check in try/except so one failure doesn't kill the entire pipeline |
| Table validation helpers | Pre-flight checks before a notebook starts its logic (do the table exists? does it has rows? does it have right columns?) |

These functions are used in NB01 through NB07.

In [0]:
import json
import traceback
from datetime import datetime, timezone
from functools import wraps
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType

## Structured Logging

Every pipeline run gets a unique `run_id` based on the UTC timestamp when the run started.
It ties together all the log entries from a single execution across notebooks.
If the orchestrator kicks off NB01 through NB07 sequentially, they all share the same run_id.

Log entries go to two places:
1. **stdout** as `[PIPELINE_LOG] {...}` - captured by Databricks job logs, useful for real-time monitoring
2. **Delta table** `pbm_sim.pipeline_audit_log` - the permanent audit trail you can query later

Each entry records the notebook, section, outcome (SUCCESS/WARNING/ERROR/SKIPPED),
row count, timing, and a free-text detail field. The detail field is truncated at 2,000
characters to prevent a full stack trace from bloating the table.

In [0]:
# For this one, I wanted a run_id that's both unique and human-readable.
# A UUID would guarantee uniqueness but "RUN_20260915_230000" tells you
# exactly when it ran without having to look anything up.
_PIPELINE_RUN_ID = datetime.now(timezone.utc).strftime("RUN_%Y%m%d_%H%M%S")
_PIPELINE_START_TIME = datetime.now(timezone.utc)

def get_run_id():
    """Return the current pipeline run ID."""
    return _PIPELINE_RUN_ID


def log_entry(notebook, section, status, row_count=0, detail="", duration_sec=0.0):
    """
    Write one structured audit log entry.

    notebook - which notebook (e.g. "NB_03")
    section - which check (e.g. "deductible_overcharge")
    status - SUCCESS, WARNING, ERROR, or SKIPPED
    row_count - how many rows the check produced (findings, not input rows)
    detail - free-text context or error message
    duration_sec - wall-clock seconds for this section
    """
    entry = {
        "run_id": _PIPELINE_RUN_ID,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "notebook": notebook,
        "section": section,
        "status": status,
        "row_count": row_count,
        "detail": detail[:2000],  # Truncate - a full stack trace can be thousands of chars
        "duration_sec": round(duration_sec, 2),
    }

    # Print to stdout so Databricks job logs capture it in real time
    print(f"[PIPELINE_LOG] {json.dumps(entry)}")

    # Write to Delta for the permanent queryable audit trail
    try:
        log_df = spark.createDataFrame([entry])
        (log_df.write
         .format("delta")
         .mode("append")
         .saveAsTable("pbm_sim.pipeline_audit_log"))
    except Exception:
        # If logging fails (table doesn't exist yet, permissions, etc),
        # we absolutely cannot let that crash the actual pipeline.
        # Print the warning and move on - the stdout log is the backup.
        print(f"[PIPELINE_LOG_WARN] Could not write audit log to Delta: {traceback.format_exc()}")


def log_summary(notebook):
    """
    Print a formatted summary of everything that happened in this notebook during this run.
    Called at the end of each notebook so the output shows a clean recap.
    """
    print(f"\n{'=' * 60}")
    print(f"  PIPELINE LOG SUMMARY: {notebook} (run: {_PIPELINE_RUN_ID})")
    print(f"{'=' * 60}")
    try:
        df_log = spark.table("pbm_sim.pipeline_audit_log")
        df_run = (
            df_log
            .filter(
                (F.col("run_id") == _PIPELINE_RUN_ID)
                & (F.col("notebook") == notebook)
            )
            .orderBy("timestamp")
        )
        for row in df_run.collect():
            icon = {"SUCCESS": "+", "WARNING": "!", "ERROR": "X", "SKIPPED": "-"}.get(row["status"], "?")
            count_str = f" ({row['row_count']} rows)" if row["row_count"] else ""
            time_str = f" [{row['duration_sec']}s]" if row["duration_sec"] else ""
            print(f"  [{icon}] {row['section']}: {row['status']}{count_str}{time_str}")
            if row["detail"] and row["status"] in ("ERROR", "WARNING"):
                # Show a truncated detail for errors and warnings so operator knows what to investigate
                print(f"      {row['detail'][:120]}")
    except Exception:
        print("  (Could not read audit log table)")
    print(f"{'=' * 60}\n")

## Error Handling Decorator

This is the core safety mechanism. In a reconciliation pipeline, each notebook runs
multiple independent checks. Take NB03 (accumulator reconciliation) - it checks for
deductible overcharges AND OOPM overcharges. If the deductible check throws an exception
(maybe a column was renamed upstream), you still want the OOPM check to run and produce
its findings. Without this decorator, an unhandled exception in the deductible check
would kill the notebook, and you'd lose the OOPM results too.

The decorator does three things on failure:
1. Catches the exception and logs it as ERROR with the full traceback
2. Returns an empty DataFrame with the expected schema so downstream unions don't break
3. Lets the notebook continue to the next section

On success, it logs the section as SUCCESS with the row count and wall-clock time.

**Usage in a downstream notebook:**
```python
@pipeline_section("NB_03", "deductible_overcharge", finding_schema=convergence_schema)
def run_deductible_check():
    # ... compute df ...
    return df

df_result = run_deductible_check()
# If the check fails, df_result is an empty DataFrame with the right columns
# so the union at the end of the notebook still works
```

In [0]:
def pipeline_section(notebook, section, finding_schema=None):
    """
    Decorator for wrapping a pipeline section with error handling and logging.

    notebook - source notebook name for the log
    section - section/check name for the log
    finding_schema - optional StructType; if the function fails, an empty DataFrame
                     with this schema is returned so downstream unions don't break
    """
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            import time
            start = time.time()
            try:
                result = func(*args, **kwargs)
                duration = time.time() - start
                row_count = result.count() if hasattr(result, "count") else 0
                log_entry(notebook, section, "SUCCESS",
                          row_count=row_count, duration_sec=duration)
                return result
            except Exception as e:
                duration = time.time() - start
                error_detail = f"{type(e).__name__}: {str(e)}\n{traceback.format_exc()}"
                log_entry(notebook, section, "ERROR",
                          detail=error_detail, duration_sec=duration)
                print(f"[PIPELINE_ERROR] {notebook}/{section}: {e}")
                # Return an empty DataFrame with the expected schema so unions don't break.
                # If no schema was provided, return None and let the caller handle it.
                if finding_schema is not None:
                    return spark.createDataFrame([], finding_schema)
                return None
        return wrapper
    return decorator

## Table Validation Helpers (Pre-flight Checks)

Before a notebook starts its actual logic, it should verify that the tables it depends on
exist and have data. Without this, a missing table produces a cryptic *AnalysisException*
50 cells into the notebook, and you spend 10 minutes figuring out that the problem was
simply that NB00 didn't run.

These are the pre-flight checks:
- **The Table exists?** Catches the case where the upstream notebook didn't run or the table got dropped
- **Are the table empty?** The table exists but has zero rows - maybe the data generator ran but the filter
  produced nothing. This is different from "table doesn't exist" and needs a different response
- **Are the required columns present?** Schema drift - a column was renamed or removed upstream. Better to catch
  this up front with a clear message than to get a "Column 'xyz' not found" error mid-analysis

In [0]:
def validate_table_exists(table_name, notebook="UNKNOWN"):
    """
    Check if a Delta table exists. Returns True/False.
    Logs a WARNING if the table is missing so the audit trail shows it.
    """
    try:
        spark.table(table_name)
        return True
    except Exception:
        log_entry(notebook, f"validate_{table_name}", "WARNING",
                  detail=f"Table {table_name} does not exist")
        print(f"[PIPELINE_WARN] Table {table_name} not found")
        return False


def validate_table_not_empty(table_name, notebook="UNKNOWN"):
    """
    Check if a Delta table has at least one row. Returns True/False.
    Logs a WARNING if the table is empty - this is a different problem than
    the table not existing. The upstream job ran but produced nothing.
    """
    try:
        df = spark.table(table_name)
        count = df.count()
        if count == 0:
            log_entry(notebook, f"validate_{table_name}", "WARNING",
                      detail=f"Table {table_name} exists but is empty")
            print(f"[PIPELINE_WARN] Table {table_name} is empty")
            return False
        return True
    except Exception as e:
        log_entry(notebook, f"validate_{table_name}", "ERROR",
                  detail=f"Cannot read {table_name}: {e}")
        return False


def validate_required_columns(df, required_cols, source, notebook="UNKNOWN"):
    """
    Check that a DataFrame has all required columns. Returns a list of missing
    column names (empty list = all present).

    This catches schema drift - if the upstream table renamed a column from
    "deductible_applied" to "ded_applied", we want to know before the join
    fails with a cryptic error.
    """
    actual = set(df.columns)
    missing = [c for c in required_cols if c not in actual]
    if missing:
        log_entry(notebook, f"validate_columns_{source}", "WARNING",
                  detail=f"Missing columns in {source}: {missing}")
        print(f"[PIPELINE_WARN] {source} missing columns: {missing}")
    return missing

## Pipeline Metadata

Convenience function for embedding run metadata into output tables.
When NB07 unions all findings, each row can carry the run_id so you can
trace any finding back to the exact pipeline execution that produced it.

In [0]:
def get_pipeline_metadata():
    """Return metadata dict for embedding in output tables or logs."""
    return {
        "run_id": _PIPELINE_RUN_ID,
        "start_time": _PIPELINE_START_TIME.isoformat(),
        "pipeline_version": "2.0",
    }

print(f"Pipeline utilities loaded. Run ID: {_PIPELINE_RUN_ID}")

Pipeline utilities loaded. Run ID: RUN_20261017_174136
